# Exercise 12 — Market efficiency

Lecture 12 made two claims that sound contradictory. At short horizons, returns are
close to unpredictable. At long horizons, the dividend-price ratio forecasts them.
This lab produces both results from the same century of data and asks how they fit
together.

Data: **Amit Goyal and Ivo Welch, updated predictor database through December 2025**
(`Data2025.xlsx`, sheet `Monthly`). Source:
[https://sites.google.com/view/agoyal145](https://sites.google.com/view/agoyal145).
The S&P 500 return series `ret` begins in January 1926, which is where our common
sample starts.

Run **Runtime → Restart and run all** before you begin, and again whenever a cell
behaves oddly.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox

from exercise_utils import FHNW, setup_style
setup_style()

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

### The data

One import cell for the whole lab. `ret` is the monthly S&P 500 total return, `Rfree`
the monthly risk-free rate, `d12` the trailing twelve-month dividend and `price` the
index level. From these we build three series:

- `lrx` — the monthly **log excess return**, $r_t = \ln(1+R_t) - \ln(1+R^f_t)$. Logs
  because they add across months, which is what a long-horizon return needs.
- `dp` — the **dividend-price ratio** $D_t/P_t$, in levels, exactly the predictor in
  Cochrane's table on slide 32.
- `dg` — monthly **log dividend growth**, for the optional Task 4.

`SAMPLE` is the one place the sample period is set. Everything downstream reads it.

In [ ]:
URL = ("https://raw.githubusercontent.com/KroeTiA/Investments/main/"
       "Exercise_12/data/Data2025.xlsx")

raw = pd.read_excel(URL, sheet_name="Monthly")
raw.index = pd.PeriodIndex(raw["yyyymm"].astype(str), freq="M")

ret = raw["ret"].astype(float)                       # total return, simple
rf = raw["Rfree"].astype(float)                      # risk-free rate, simple
lrx = np.log(1 + ret) - np.log(1 + rf)               # log excess return
dp = (raw["d12"] / raw["price"]).astype(float)       # dividend-price ratio, level
dg = np.log(raw["d12"].astype(float)).diff()         # log dividend growth

SAMPLE = ("1926-01", "2025-12")     # one century of returns — the common sample
# SAMPLE = ("1947-01", "2004-12")   # Cochrane (2008) window — uncomment to reproduce slide 32

lo, hi = SAMPLE
print(f"sample {lo} to {hi}:  {len(ret.loc[lo:hi])} months, "
      f"{len(ret.loc[lo:hi]) / 12:.0f} years")
print(f"mean excess return {lrx.loc[lo:hi].mean() * 12:6.2%} p.a.   "
      f"volatility {lrx.loc[lo:hi].std() * np.sqrt(12):6.2%} p.a.")
print(f"D/P: mean {dp.loc[lo:hi].mean():.4f}   "
      f"min {dp.loc[lo:hi].min():.4f}   max {dp.loc[lo:hi].max():.4f}")

## Task 1 — How much do past returns tell you?

Two descriptive statistics, both weak-form tests, both on the common sample.

**(a)** Compute the autocorrelations $\rho_1, \dots, \rho_{12}$ of the monthly
*total* return `ret` and plot them as a bar chart with the $\pm 1.96/\sqrt{T}$ band.
Then run a Ljung-Box test on the first twelve lags.

**(b)** The variance ratio compares the variance of a $q$-month return to $q$ times
the variance of a one-month return:

$$VR(q) \;=\; \frac{\mathrm{Var}\!\left(\sum_{j=1}^{q} r_{t+j}\right)}{q \cdot \mathrm{Var}(r_t)}$$

Under the random walk it equals one at every $q$. Compute it on `lrx` for horizons of
1, 2, 3, 5 and 8 years, and normalise each value by the one-year figure, as Campbell
does in the table on slide 21.

**Deliverable.** The autocorrelation figure, the Ljung-Box p-value, and a variance-ratio
table with the raw and normalised values next to the asymptotic standard error
$\sqrt{2(2q-1)(q-1) / (3qT)}$.

Then answer, in one line each: what fraction of the variance of next month's return does
$\rho_1$ account for, and is the pattern in the variance ratios larger than its own
standard errors?

In [ ]:
r1 = ret.loc[lo:hi]
T = len(r1)
# TODO: autocorrelations at lags 1-12, and the Ljung-Box test on all twelve
band = 1.96 / np.sqrt(T)

In [ ]:
fig1, ax = plt.subplots(figsize=(7.5, 4.0))
# TODO: bar chart of rho(1)...rho(12) with the +/- band as dashed lines
ax.set_xlabel("lag, months")
ax.set_ylabel("autocorrelation of monthly return")
ax.set_xticks(np.arange(1, 13))
plt.show()

In [ ]:
x1 = lrx.loc[lo:hi]
HORIZONS = [(1, 12), (2, 24), (3, 36), (5, 60), (8, 96)]
def variance_ratio(x, q):
    x = np.asarray(x, float)
    n = len(x)
    mu = x.mean()
    # TODO: one-period variance, then the variance of the overlapping q-period sums
rows = {}
for years, q in HORIZONS:
    label = f"{years} year" + ("s" if years > 1 else "")
    # TODO: the ratio itself, and the asymptotic standard error given in the task
    rows[label] = {"raw VR": vr, "asym. SE": se, "z": (vr - 1) / se}
vr_tab = pd.DataFrame(rows).T
# TODO: add a "normalised" column: each raw VR divided by the one-year value
print(vr_tab.to_string(float_format=lambda v: f"{v:7.3f}"))

In [ ]:
fig2, ax = plt.subplots(figsize=(7.5, 4.0))
yrs = [y for y, _ in HORIZONS]
# TODO: plot the normalised variance ratio against the horizon, with a +/- 2 SE band
ax.set_xlabel("horizon, years")
ax.set_ylabel("normalised variance ratio")
fig2.tight_layout()
plt.show()

## Task 2 — The same question at longer horizons

Now use a predictor. Regress the excess return realised over the next $h$ months on
today's dividend-price ratio:

$$\sum_{j=1}^{h} r_{t+j} \;=\; a + b \cdot (D_t/P_t) + \varepsilon_{t+h}$$

Run it for $h = 1, 3, 12, 36$ and $60$ months on the common sample and report $b$,
$R^2$, and **two** t-statistics for $b$: ordinary OLS, and Newey-West.

**New this week: HAC standard errors.** Consecutive observations of a five-year return
share 59 of their 60 months, so the residuals are heavily autocorrelated and OLS
standard errors are far too small. Newey-West standard errors correct for that. The
call is one keyword on the fit you already know:

    sm.OLS(y, sm.add_constant(x)).fit(cov_type="HAC", cov_kwds={"maxlags": h})

**Deliverable.** A table with one row per horizon: $b$, t(OLS), t(NW), $R^2$, and the
$R^2$ rescaled to an annual basis, $R^2 \times 12/h$. Plus a figure of the last two
columns against the horizon.

Then answer: the $R^2$ rises by a factor of fifty from one month to five years. Does the
forecast get fifty times better?

*If your one-month $R^2$ comes out near 0.9, the shift has gone the wrong way and you
are regressing the past on the present. Use log returns for anything summed across
months; the predictor stays in levels.*

In [ ]:
LADDER = [("1 month", 1), ("3 months", 3), ("1 year", 12),
          ("3 years", 36), ("5 years", 60)]
def forward_sum(x, h):
    ...
    # TODO: roll h periods, then shift the window into the future
def predictive(y_src, x_src, h):
    y = forward_sum(y_src, h).rename("y")
    df = pd.concat([y, x_src.rename("x")], axis=1).dropna()
    X = sm.add_constant(df["x"])
    # TODO: fit twice — once plain, once with HAC errors at maxlags = h
    return {"b": ols.params.iloc[1],
            "t(OLS)": ols.tvalues.iloc[1],
            "t(NW)": nw.tvalues.iloc[1],
            "R2": ols.rsquared,
            "R2 x 12/h": ols.rsquared * 12 / h,
            "N": len(df)}
rows = {name: predictive(lrx.loc[lo:hi], dp.loc[lo:hi], h)
        for name, h in LADDER}
ladder = pd.DataFrame(rows).T
ladder["N"] = ladder["N"].astype(int)
print(f"log excess returns on D/P, {lo} to {hi}\n")
print(ladder.to_string(float_format=lambda v: f"{v:9.3f}"))

In [ ]:
fig3, ax = plt.subplots(figsize=(7.5, 4.0))
months = [h for _, h in LADDER]
# TODO: plot R2 and R2 x 12/h against the horizon in months
ax.set_xlabel("forecast horizon h, months")
ax.set_ylabel("share of variance explained")
ax.set_ylim(0, None)
ax.legend()
plt.show()

## Task 3 — Would you have believed it in 1975?

Split the common sample into two halves of equal length and run the one-year and
five-year regressions again, separately, in each half.

**Deliverable.** A table with one row per half and per horizon, carrying $b$, t(NW),
$R^2$ and $N$. Plus a two-panel scatter of the five-year forward excess return against
$D_t/P_t$, one panel per half, each with its fitted line.

Reuse `predictive()` from Task 2 rather than rewriting it — only the slice changes.

Then answer two questions. First: an investor in 1975 fits this regression on everything
available and finds the first half's result. What allocation decision does it support,
and what happened to it over the next fifty years? Second: the two halves are the same
length, use the same predictor and the same specification. Name the one thing that
differs, and say whether it makes the full-sample estimate trustworthy or not.

In [ ]:
span = ret.loc[lo:hi].index
mid = len(span) // 2
HALVES = [("first half", lo, str(span[mid - 1])),
          ("second half", str(span[mid]), hi)]
print(f"splitting {lo}..{hi} at {span[mid]}\n")
rows = {}
# TODO: rerun predictive() at h = 12 and h = 60 inside each half of HALVES
halves = pd.DataFrame(rows).T[["b", "t(NW)", "R2", "N"]]
halves["N"] = halves["N"].astype(int)
print(halves.to_string(float_format=lambda v: f"{v:9.3f}"))

In [ ]:
fig4, axes = plt.subplots(1, 2, figsize=(9.5, 4.2), sharey=True)
for ax, (label, a, b_) in zip(axes, HALVES):
    y5 = forward_sum(lrx.loc[a:b_], 60).rename("y")
    df = pd.concat([y5, dp.loc[a:b_].rename("x")], axis=1).dropna()
    # TODO: scatter the five-year forward return on D/P and add the fitted line
    ax.set_title(f"{label}: {a} to {b_}", fontsize=10)
    ax.set_xlabel("D/P at date t")
axes[0].set_ylabel("log excess return over the next 5 years")
plt.show()

## Task 4 — Optional: the dog that did not bark

The present-value identity says $D/P$ must forecast either returns or dividend growth.
Task 2 showed the return half. Run the other half: regress log dividend growth over the
next $h$ months on $D_t/P_t$, at $h = 12$ and $h = 60$.

Run it twice — once on the common sample, and once on 1947 to 2025. The two answers
disagree. Work out which feature of the earlier period is responsible, and what that
implies for Cochrane's argument.

**Deliverable.** The two tables side by side, and one sentence naming the cause.

In [ ]:
WINDOWS = [("1926-2025", "1926-01", "2025-12"),
           ("1947-2025", "1947-01", "2025-12")]
rows = {}
# TODO: run the dividend-growth regression on both windows at h = 12 and h = 60
dog = pd.DataFrame(rows).T[["b", "t(NW)", "R2", "N"]]
dog["N"] = dog["N"].astype(int)
print("log dividend growth on D/P\n")
print(dog.to_string(float_format=lambda v: f"{v:9.3f}"))

## This week, off-class

**Drill 12** and **Transfer 12** are in Moodle and close on Sunday at 23:59. The
transfer items use the numbers you produced above, so keep this notebook open while
you answer them.